### 환경설정

In [ ]:
# Open AI API key 등록
import os

OPENAI_API_KEY='API_KEY'

# 현재 노트북 커널에 환경변수 등록
os.environ['OPENAI_API_KEY']=OPENAI_API_KEY

# 랭스미스로 기록 할 건지 여부
LANGSMITH_TRACING="true"

# 랭스미스 사이트에서 기록 할 나만의 url
LANGSMITH_ENDPOINT="https://api.smith.langchain.com"

# 내 랭스미스 url key
LANGSMITH_API_KEY="API_KEY"

# 내 계정의 프로젝트이름
LANGSMITH_PROJECT="langchain0422"

# 현재 노트북 커널에 환경변수 등록
os.environ['LANGSMITH_TRACING']=LANGSMITH_TRACING
os.environ['LANGSMITH_ENDPOINT']=LANGSMITH_ENDPOINT
os.environ['LANGSMITH_API_KEY']=LANGSMITH_API_KEY
os.environ['LANGSMITH_PROJECT']=LANGSMITH_PROJECT

In [2]:
# 라이브러리 설치하기

!pip install langchain langchain-openai langchain-community faiss-cpu pypdf chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180

### 네이버 뉴스기사를 기반으로 답변하는 챗봇 만들기
 - 필요시 크롤링 기술이나 크롤링 에이전트 등을 통해서 뉴스 URL을 수집
 - 현재는 직접 URL을 선정하고 챗봇에서 삽입하는 방식으로 진행

In [6]:
# ** 웹문서 RAG 도구 **

# 웹문서 파싱 도구
import bs4

# 웹문서 로딩 도구
from langchain_community.document_loaders import WebBaseLoader

# 재귀적 텍스트 분할 도구
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 임베딩 도구
from langchain_openai import OpenAIEmbeddings

# 벡터 데이터 베이스 도구
from langchain_community.vectorstores import FAISS


# ** Chain 구성을 위한 도구 **

# 프롬프트 템플릿
from langchain_core.prompts import ChatPromptTemplate

# 모델을 생성하는 도구
from langchain.chat_models import init_chat_model

# 문자열 파싱 도구
from langchain_core.output_parsers import StrOutputParser

# 데이터 전달용 Runnable
from langchain_core.runnables import RunnablePassthrough

### 1. 뉴스기사 파싱하기

In [7]:
# 파싱할 URL
url = 'https://www.dt.co.kr/article/12059603?ref=naver'

In [13]:
# 웹페이지를 읽어주는 로더 생성
web_loader = WebBaseLoader(
    web_path = [url],   # 읽을 웹페이지 경로(여러개 가능)
    bs_kwargs = dict(
        parse_only = bs4.SoupStrainer(
            'div', attrs = {'class' : 'view'}
        )
    )
)

In [14]:
# 문서로딩
docs = web_loader.load()
docs

[Document(metadata={'source': 'https://www.dt.co.kr/article/12059603?ref=naver'}, page_content='\n‘시간은 내 편’ 동상이몽…“전쟁도 평화도 아닌 교착상태”   도널드 트럼프 미국 대통령 [AP=연합뉴스] “미국이냐 이란이냐.” 누가 더 오래 버티냐의 진짜 싸움이 시작됐다는 분석이 나오고 있다.지난 주말 파키스탄에서 열릴 것으로 기대를 모은 미국과 이란의 2차 종전 협상이 양측의 입장차로 결국 무산됐다.이란은 이미 호르무즈 해협 봉쇄 카드로 미국을 포함한 세계 경제에 유가 급등이라는 큰 고통을 가하고 있다. 미국도 이에 맞서 이란 해상 전면 봉쇄라는 강경 카드로 맞불을 놓은 상황이다.미국과 이란 모두 ‘시간은 내 편’이라는 인식 속에서 ‘누가 더 경제적 고통을 오래 버티느냐’ 싸움에 들어감에 따라 전쟁도 평화도 아닌 현재의 교착 상태가 길어질 수 있다는 우려가 나온다.27일(현지시간) 외신들에 따르면, 지난 주말 파키스탄 이슬라마바드에서 추진된 2차 종전 협상은 핵심 조건에 대한 이견으로 결렬됐다. 미국은 스티브 윗코프 특사와 재러드 쿠슈너를 파견하며 합의 의지를 보였고, 이란 역시 아바스 아라그치 외무부 장관이 현지에 도착하며 기대감을 높였으나 이란 내 강경파의 반대에 부딪힌 것으로 분석된다.미국은 스티브 윗코프 특사와 도널드 트럼프 대통령 맏사위 재러드 쿠슈너의 파견을 공식 발표할 만큼 합의 의지가 강했다.아바스 아라그치 이란 외무부 장관도 24일 파키스탄에 도착해 협상 기대감이 커졌다. 그러나 이란 내 의사결정을 지배하는 강경파는 핵 프로그램, 호르무즈 해협에 대한 통제권 등에 대한 기존 입장을 견지했다.이 같은 협상의 교착에서는 전쟁에서 승기를 잡은 채 휴전에 들어갔다는 이란과 미국의 공통된 인식이 관측된다.이란은 호르무즈 해협 봉쇄로, 미국은 휴전 후 이뤄진 전격적인 이란 해안 전면 봉쇄로 서로 상대에 경제적 고통을 주고 있다고 여긴다. 시간이 갈수록 상대가 점점 더 견디기 

### 2. chunk 단위로 쪼개기

In [16]:
# 구분자는 설정하지 않고 기본 값 사용 => '\n\n', '\n', ' ', ''
news_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 100,   # 청크 사이즈
    chunk_overlap = 50   # 청크 오버랩 사이즈
)

news_chunk = news_splitter.split_documents(docs)
len(news_chunk)

51

### 3. 임베딩 후 벡터DB에 저장

In [17]:
# FAISS 활용
faiss_db = FAISS.from_documents(
    news_chunk,   # 청크 데이터
    OpenAIEmbeddings()   # 임베딩 도구
)

### 4. 검색기 생성

In [19]:
# chunk 사이즈가 작기 때문에 찾아오는 유사청크 갯수를 늘려서 설정
news_retriver = faiss_db.as_retriever(search_kwagrs = {'k' : 5})

### 5. Chain 생성 및 호출

In [22]:
template = ChatPromptTemplate.from_messages([
    ('system', '''
        너는 뉴스 챗봇이야. 아래 context를 참고해서 question에 대한 답변을 만들어줘.
        모르는 정보는 모른다고 반드시 답변해.

        context :
            {context}
    '''),
    ('human', 'question : {question}')
])

In [23]:
# chain 구성
news_chain = {'question' : RunnablePassthrough(),
              'context' : news_retriver} | template | init_chat_model('openai:gpt-4o-mini') | StrOutputParser()

In [24]:
print(news_chain.invoke('최근 이란 미국 관련 뉴스 알려줘'))

최근 이란과 미국 간의 대치 상황에 대한 뉴스가 보도되고 있습니다. 뉴욕타임스는 이란과 미국이 각각 상대방보다 더 오래 버티기를 노리고 있어 세계 경제에 큰 이해관계가 걸린 상황이라고 분석했습니다. 또한, 미국 싱크탱크 전쟁연구소(ISW)는 이란 내 강경파인 이슬람혁명수비대(IRGC)가 의사결정 과정에서 큰 영향을 미치고 있으며, 이로 인해 종전 협상이 어려운 상황에 놓여 있다고 전했습니다. 미국은 스티브 윗코프 특사와 도널드 트럼프 대통령의 맏사위인 재러드 쿠슈너를 파견하기로 하며 합의 의지를 보이고 있습니다.


In [25]:
print(news_chain.invoke('최근 치킨 관련 뉴스 알려줘'))

죄송하지만, 제공된 context에는 치킨 관련 뉴스에 대한 정보가 포함되어 있지 않아서 답변할 수 없습니다.
